# 9장. Chroma 기반 RAG 실습

이 장은 PDF 교재의 `6.8 벡터스토어 vs 데이터베이스 특징 비교` 중 Chroma 내용과 `LLM/llm.ipynb`의 Chroma 기반 RAG, Chroma 서버 접근 실습을 통합한 장입니다.

## 이 장에서 다루는 내용

1. Chroma란 무엇인가
2. Chroma와 FAISS 차이
3. 로컬 Chroma RAG 구성
4. 문서 로딩과 청크 분할
5. 임베딩 모델 정의
6. Chroma 벡터스토어 생성과 저장
7. Ollama LLM 연결 확인
8. Chroma Retriever 기반 RAG Chain
9. Chroma 서버 실행
10. `HttpClient`로 Chroma 서버 연결
11. 원격/중앙 벡터 DB 접근
12. 인터랙티브 질의 루프
13. Chroma 사용 시 오류와 해결 방법

8장에서 FAISS를 사용했다면, 이번 장에서는 개발 환경에서 사용하기 쉬운 영속성 벡터 DB인 Chroma를 사용합니다.


## 9.1 Chroma란?

Chroma는 개발자가 빠르게 시작하기 좋은 벡터 데이터베이스입니다.

PDF 6.8의 Chroma 설명은 다음과 같습니다.

| 항목 | 설명 |
|---|---|
| 주요 특징 | 영속성 벡터 DB, 빠른 시작과 사용 편의성, SQLite 사용 |
| 특장점 | 개발 환경에 적합한 경량 솔루션 |
| 주요 기능 | HNSW 기반 최근접 이웃 검색, 코사인 유사도 기본 지원 |

Chroma는 FAISS보다 DB에 가까운 사용성을 제공합니다. 문서, 임베딩, 메타데이터를 저장하고 다시 로드하기 쉽습니다.


## 9.2 Chroma vs FAISS

| 항목 | FAISS | Chroma |
|---|---|---|
| 성격 | 고성능 벡터 검색 라이브러리 | 경량 벡터 데이터베이스 |
| 영속 저장 | 별도 저장 처리 필요 | `persist_directory`로 쉽게 저장 |
| 사용 편의성 | 검색 성능 중심 | 개발 편의성 중심 |
| 메타데이터 관리 | 상대적으로 직접 관리 필요 | 문서와 메타데이터 관리가 쉬움 |
| 서버 모드 | 기본은 라이브러리 방식 | 로컬/서버 방식 모두 가능 |
| 실습 적합성 | 빠른 벡터 검색 이해에 좋음 | RAG 앱 개발 흐름에 좋음 |

정리하면 FAISS는 검색 라이브러리 느낌이 강하고, Chroma는 개발자가 바로 쓰기 편한 벡터 DB 느낌이 강합니다.


## 9.3 설치 준비

`llm.ipynb`의 Chroma RAG 예제 설치 주석은 다음과 같습니다.

```text
pip install langchain langchain-community chromadb sentence-transformers langchain-text-splitters
```

필요 패키지 역할:

| 패키지 | 역할 |
|---|---|
| `chromadb` | Chroma 벡터 데이터베이스 |
| `sentence-transformers` | 임베딩 모델 실행 |
| `langchain-community` | Chroma, TextLoader, Ollama 연동 |
| `langchain-core` | Prompt, Runnable, OutputParser |
| `langchain-text-splitters` | 문서 청크 분할 |


In [ ]:
# Chroma RAG 실습용 설치 예시입니다.
# 필요한 경우 주석을 해제하고 실행하세요.

# %pip install langchain langchain-community langchain-core langchain-text-splitters
# %pip install chromadb sentence-transformers


## 9.4 로컬 Chroma RAG: 모듈 import

원본 노트북의 Chroma RAG 예제는 다음 모듈을 사용합니다.

| 모듈 | 역할 |
|---|---|
| `os` | 파일/폴더 경로 확인 |
| `shutil` | 기존 벡터 저장소 폴더 삭제 |
| `ChatPromptTemplate` | 프롬프트 구성 |
| `StrOutputParser` | LLM 출력 문자열 파싱 |
| `RunnablePassthrough` | 질문 원문 전달 |
| `TextLoader` | 텍스트 파일 로드 |
| `Chroma` | Chroma 벡터스토어 |
| `HuggingFaceEmbeddings` | 임베딩 모델 |
| `Ollama` | 답변 생성 LLM |
| `RecursiveCharacterTextSplitter` | 문서 청크 분할 |


In [ ]:
# 파일과 폴더 경로 처리를 위해 사용합니다.
import os

# 기존 Chroma 저장소 폴더 삭제에 사용합니다.
import shutil

# RAG 체인 구성에 필요한 LangChain 핵심 모듈입니다.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 문서 로더와 벡터스토어입니다.
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma

# 임베딩 모델과 LLM입니다.
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama

# 문맥이 가능한 한 유지되도록 문서를 청크로 분할합니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter


## 9.5 기본 설정

원본 노트북은 다음 설정값을 사용합니다.

| 설정 | 값 | 설명 |
|---|---|---|
| `TEXT_FILE_PATH` | `./dataset/history.txt` | 로드할 텍스트 파일 |
| `PERSIST_DIRECTORY` | `./chroma_store` | ChromaDB를 저장할 로컬 디렉토리 |
| `OLLAMA_BASE_URL` | `http://localhost:11434` | Ollama 로컬 서버 주소 |
| `OLLAMA_MODEL_NAME` | `gemma2` | 사용할 Ollama 모델 |

상대 경로를 사용하므로 노트북 실행 위치가 중요합니다. 원본 기준으로는 `C:/work/LLM`에서 실행하는 것을 가정합니다.


In [ ]:
# 로드할 텍스트 파일 경로입니다.
TEXT_FILE_PATH = "./dataset/history.txt"

# ChromaDB를 저장할 로컬 디렉토리입니다.
PERSIST_DIRECTORY = "./chroma_store"

# Ollama 로컬 서버 주소입니다.
OLLAMA_BASE_URL = "http://localhost:11434"

# 사용할 Ollama 모델명입니다.
OLLAMA_MODEL_NAME = "gemma2"


## 9.6 문서 로드

Chroma RAG에서도 첫 단계는 문서 로드입니다.

원본 노트북은 `try-except`로 파일 없음이나 로드 오류를 처리합니다.

```python
loader = TextLoader(TEXT_FILE_PATH, encoding='UTF8')
documents = loader.load()
```

파일 경로가 틀리면 `FileNotFoundError`가 발생합니다.


In [ ]:
# 1. 문서 로드
print(f"'{TEXT_FILE_PATH}' 파일 로드 중...")

try:
    # 텍스트 파일을 UTF-8 인코딩으로 로드합니다.
    loader = TextLoader(TEXT_FILE_PATH, encoding='UTF8')
    documents = loader.load()
except FileNotFoundError:
    # 파일이 없을 때 안내 메시지를 출력합니다.
    print(f"오류: 파일이 존재하지 않습니다. 경로를 확인하세요: {TEXT_FILE_PATH}")
    raise
except Exception as e:
    # 그 외 로드 오류를 처리합니다.
    print(f"파일 로드 중 오류 발생: {e}")
    raise

print("로드된 문서 수:", len(documents))


## 9.7 문서 청크 분할

원본 Chroma 예제는 `RecursiveCharacterTextSplitter`를 사용합니다.

```python
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)
texts = text_splitter.split_documents(documents)
```

각 설정의 의미:

| 설정 | 의미 |
|---|---|
| `chunk_size=500` | 청크 하나의 최대 길이를 500자로 설정합니다. |
| `chunk_overlap=50` | 청크 사이에 50자 중복을 둡니다. |
| `length_function=len` | Python `len()`으로 길이를 계산합니다. |

overlap을 두면 문맥이 청크 경계에서 끊기는 문제를 줄일 수 있습니다.


In [ ]:
# 문서 분할기를 생성합니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

# 로드한 문서를 청크 목록으로 분할합니다.
texts = text_splitter.split_documents(documents)

# 청크 수를 확인합니다.
print(f"총 {len(texts)}개의 청크로 분할되었습니다.")


## 9.8 임베딩 모델 정의

원본 노트북은 Hugging Face 임베딩 모델을 사용합니다.

```python
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
```

이 모델은 문장을 벡터로 바꾸는 sentence-transformers 모델입니다.

주의:

- 처음 실행하면 모델 다운로드가 필요할 수 있습니다.
- 한국어 검색 품질을 더 높이려면 한국어/다국어 임베딩 모델을 고려할 수 있습니다.


In [ ]:
# 임베딩 모델을 정의합니다.
print("임베딩 모델 정의")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


## 9.9 ChromaDB 벡터 저장소 생성

원본 노트북은 기존 `./chroma_store` 폴더가 있으면 삭제하고 새로 만듭니다.

```python
if os.path.exists(PERSIST_DIRECTORY):
    shutil.rmtree(PERSIST_DIRECTORY)
```

그 다음 Chroma 벡터스토어를 생성합니다.

```python
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY
)
vectordb.persist()
```

`persist_directory`를 지정하면 벡터 저장소가 로컬 폴더에 저장됩니다.


In [ ]:
# 기존 벡터 저장소가 있으면 삭제합니다.
if os.path.exists(PERSIST_DIRECTORY):
    print(f"'{PERSIST_DIRECTORY}' 기존 벡터 저장소 삭제 중...")
    shutil.rmtree(PERSIST_DIRECTORY)

# ChromaDB 벡터 저장소를 생성하고 저장합니다.
print(f"ChromaDB 벡터 저장소 생성 및 '{PERSIST_DIRECTORY}'에 저장 중...")

vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY
)

# 현재 버전의 Chroma/LangChain에서는 자동 persist되는 경우도 있지만, 원본 노트북 흐름대로 호출합니다.
vectordb.persist()

print("벡터 저장소가 성공적으로 저장되었습니다.")


## 9.10 Ollama LLM 및 Retriever 정의

원본 노트북은 Ollama 연결을 확인하기 위해 `llm.invoke("Hello")`를 먼저 실행합니다.

```python
llm = Ollama(model=OLLAMA_MODEL_NAME, base_url=OLLAMA_BASE_URL)
llm.invoke("Hello")
```

그 다음 Chroma 벡터스토어를 Retriever로 변환합니다.

```python
retriever = vectordb.as_retriever()
```


In [ ]:
# Ollama LLM 및 검색기 초기화
print(f"Ollama LLM ({OLLAMA_MODEL_NAME}) 및 검색기 초기화 중...")

try:
    # Ollama 모델을 초기화합니다.
    llm = Ollama(model=OLLAMA_MODEL_NAME, base_url=OLLAMA_BASE_URL)

    # 연결 테스트입니다.
    llm.invoke("Hello")
    print("Ollama LLM 연결 성공")
except Exception as e:
    # Ollama 서버 또는 모델 문제가 있을 때 안내합니다.
    print(f"오류: Ollama LLM에 연결할 수 없습니다. URL: {OLLAMA_BASE_URL}")
    print(f"Ollama가 실행 중인지, '{OLLAMA_MODEL_NAME}' 모델이 설치되었는지 확인하세요.")
    raise

# Chroma 벡터 저장소를 Retriever로 변환합니다.
retriever = vectordb.as_retriever()


## 9.11 Chroma 기반 RAG Chain 구성

Chroma RAG 체인도 FAISS RAG와 거의 같습니다.

```python
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

벡터스토어만 FAISS에서 Chroma로 바뀌었을 뿐, Retriever 이후의 흐름은 동일합니다.


In [ ]:
# RAG 프롬프트 템플릿입니다.
template = """
당신은 질문에 답변하는 AI 어시스턴트입니다.
제공된 context만을 바탕으로 질문에 답변하세요. 모르면 모른다고 답하세요.

[Context]
{context}

[Question]
{question}
"""

# 프롬프트 객체를 생성합니다.
prompt = ChatPromptTemplate.from_template(template)

# Chroma Retriever 기반 RAG 체인을 구성합니다.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


## 9.12 Chroma RAG 질의 실행

원본 노트북의 질문은 다음과 같습니다.

```text
고조선은 언제 설립되었는지 알려줘.
```

이 질문은 history.txt 문서에서 관련 내용을 검색한 뒤 Ollama가 답변합니다.


In [ ]:
# RAG 체인 실행
query = "고조선은 언제 설립되었는지 알려줘."

try:
    # Chroma 기반 RAG 체인을 실행합니다.
    response = rag_chain.invoke(query)

    # 질문과 답변을 출력합니다.
    print("[질문]:", query)
    print("[답변]:", response)
except Exception as e:
    # 오류를 출력합니다.
    print(f"RAG 체인 실행 중 오류가 발생했습니다: {e}")


## 9.13 Chroma 서버 방식 개요

`llm.ipynb`에는 Chroma를 로컬 폴더로 쓰는 방식뿐 아니라 Chroma 서버에 접속하는 예제도 있습니다.

서버 방식은 다음 상황에 유용합니다.

- 여러 사용자가 같은 벡터 DB를 공유해야 할 때
- 중앙 서버에 문서 벡터를 저장하고 각자 로컬 LLM으로 답변을 만들 때
- 벡터 DB를 애플리케이션 서버와 분리하고 싶을 때

원본 노트북의 Chroma 서버 실행 명령:

```powershell
chroma run --host 0.0.0.0 --port 8000 --path ./chroma_store
```

서버 주소와 포트:

```python
SERVER_HOST = "localhost"
SERVER_PORT = 8000
```


In [ ]:
# Chroma 서버 실행 명령 예시입니다.
# 이 명령은 노트북 셀이 아니라 PowerShell 또는 터미널에서 실행하세요.

# chroma run --host 0.0.0.0 --port 8000 --path ./chroma_store


## 9.14 Chroma 서버 접속 모듈 import

원본 노트북의 서버 접근 예제는 다음 모듈을 사용합니다.

| 모듈 | 역할 |
|---|---|
| `time` | 응답 시간 측정 |
| `Chroma` | LangChain Chroma 벡터스토어 연결 |
| `SentenceTransformerEmbeddings` | 서버 벡터 DB와 같은 임베딩 모델 사용 |
| `Ollama` | 로컬 LLM 답변 생성 |
| `ChatPromptTemplate` | 프롬프트 구성 |
| `StrOutputParser` | 응답 문자열 파싱 |
| `RunnablePassthrough` | 질문 전달 |
| `chromadb.HttpClient` | Chroma 서버와 통신 |


In [ ]:
# 응답 시간 측정에 사용합니다.
import time

# Chroma 서버 컬렉션을 LangChain 벡터스토어로 연결합니다.
from langchain_community.vectorstores import Chroma

# 구 LangChain 방식의 SentenceTransformer 임베딩입니다.
from langchain_community.embeddings import SentenceTransformerEmbeddings

# ChromaDB 서버에 접속하기 위한 클라이언트입니다.
from chromadb import HttpClient


## 9.15 서버 정보와 로컬 LLM 설정

원본 노트북은 중앙 벡터 DB 서버와 로컬 Ollama LLM을 분리해서 설정합니다.

```python
SERVER_HOST = "localhost"
SERVER_PORT = 8000
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma2"
```

이 구조에서는 Chroma 서버는 중앙 벡터 DB 역할을 하고, Ollama는 각 사용자 PC에서 답변 생성 모델 역할을 합니다.


In [ ]:
# 중앙 벡터 DB 서버 정보입니다.
SERVER_HOST = "localhost"
SERVER_PORT = 8000

# 로컬 Ollama LLM 설정입니다.
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma2"


## 9.16 Chroma 서버 연결 테스트

`HttpClient`로 Chroma 서버에 연결하고, `heartbeat()`로 연결 상태를 확인합니다.

```python
client = HttpClient(host=SERVER_HOST, port=SERVER_PORT)
client.heartbeat()
```

연결에 실패하면 Chroma 서버가 실행 중인지, IP와 포트가 맞는지 확인해야 합니다.


In [ ]:
# 중앙 벡터 DB 서버 연결 테스트
print(f"[접속 시도] 중앙 벡터 서버 ({SERVER_HOST}:{SERVER_PORT}) 연결 중...")

try:
    # 원격 Chroma 서버 사용을 선언합니다.
    client = HttpClient(host=SERVER_HOST, port=SERVER_PORT)

    # 서버 연결 상태를 확인합니다.
    client.heartbeat()
    print("서버 연결 성공")
except Exception as e:
    print(f"서버 연결 실패. IP({SERVER_HOST})와 포트({SERVER_PORT})를 확인하세요.")
    print(e)
    raise


## 9.17 서버 벡터 DB와 같은 임베딩 모델 사용

서버에 저장된 벡터를 검색하려면, 검색할 때 사용하는 임베딩 모델이 벡터 DB 생성 시 사용한 모델과 같아야 합니다.

원본 노트북:

```python
embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
```

주석에는 다른 추천 모델로 `BAAI/bge-m3`도 언급되어 있습니다.

임베딩 모델이 다르면 질문 벡터와 저장된 문서 벡터가 다른 공간에 놓이므로 검색 품질이 크게 떨어질 수 있습니다.


In [ ]:
# 벡터 DB 생성 시 사용한 모델과 동일한 임베딩 모델을 지정해야 합니다.
# 기타 추천 모델 예시: HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
server_embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


## 9.18 원격 Chroma 벡터 저장소 연결

원본 노트북은 Chroma 서버의 컬렉션 이름을 `langchain`으로 사용합니다.

```python
vector_store = Chroma(
    client=client,
    collection_name="langchain",
    embedding_function=embeddings,
)
```

Chroma에서 `collection_name`은 벡터 데이터 묶음의 이름입니다. 최초 벡터 저장소 생성 시 collection name을 지정하지 않으면 `langchain` 이름이 자동 저장될 수 있습니다.


In [ ]:
# 원격 Chroma 벡터 저장소에 연결합니다.
vector_store = Chroma(
    client=client,
    collection_name="langchain",
    embedding_function=server_embeddings,
)

# Retriever로 변환합니다.
server_retriever = vector_store.as_retriever()

print("벡터 저장소 연결 완료")


## 9.19 서버형 Chroma RAG Chain 구성

서버형 Chroma에서도 RAG 체인 구조는 동일합니다.

```text
질문 -> Chroma 서버 Retriever -> Prompt -> 로컬 Ollama -> 문자열 응답
```

벡터 검색은 Chroma 서버가 수행하고, 답변 생성은 로컬 Ollama가 수행합니다.


In [ ]:
# 로컬 Ollama LLM을 설정합니다.
server_llm = Ollama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)

# 서버형 RAG 프롬프트입니다.
server_template = """질문에 대해 제공된 Context만을 기반으로 답변하세요.
Context: {context}
Question: {question}
Answer:"""

# 프롬프트 객체를 생성합니다.
server_prompt = ChatPromptTemplate.from_template(server_template)

# 서버형 Chroma RAG 체인을 구성합니다.
server_rag_chain = (
    {"context": server_retriever, "question": RunnablePassthrough()}
    | server_prompt
    | server_llm
    | StrOutputParser()
)


## 9.20 인터랙티브 질의 루프

원본 노트북은 사용자가 `exit`, `quit`, `종료`를 입력할 때까지 질문을 반복해서 받는 루프를 제공합니다.

주요 기능:

- 사용자 입력 받기
- 종료 명령 처리
- 빈 입력 무시
- 응답 생성 시간 측정
- 오류 발생 시 메시지 출력

Jupyter Notebook에서 `input()` 루프를 실행하면 셀이 계속 대기하므로, 필요할 때만 실행하는 것이 좋습니다.


In [ ]:
# 질의 응답 인터랙티브 루프입니다.
print("[준비 완료] 질의를 시작합니다. 종료하려면 'exit' 입력")

while True:
    # 사용자 질문을 입력받습니다.
    query = input("\n질문 입력 > ")

    # 종료 명령을 처리합니다.
    if query.lower() in ["exit", "quit", "종료"]:
        print("질의 세션을 종료합니다.")
        break

    # 빈 입력은 건너뜁니다.
    if not query.strip():
        continue

    # 응답 시간을 측정합니다.
    start_time = time.time()
    print("답변 생성 중...", end="", flush=True)

    try:
        # RAG 체인을 실행합니다.
        response = server_rag_chain.invoke(query)
        elapsed = time.time() - start_time
        print(f"\r[답변] ({elapsed:.2f}초 소요)\n{response}\n")
    except Exception as e:
        print(f"\n[오류 발생] {e}")


## 9.21 Chroma 사용 시 개선 포인트

### 1. collection name 명시

여러 프로젝트를 한 Chroma 서버에서 다룬다면 `collection_name`을 명확히 지정하는 것이 좋습니다.

### 2. metadata 활용

문서 출처, 페이지 번호, 파일명 등을 metadata에 저장하면 답변 출처 표시가 쉬워집니다.

### 3. 임베딩 모델 통일

문서를 저장할 때와 검색할 때 같은 임베딩 모델을 사용해야 합니다.

### 4. persist directory 관리

실습에서는 기존 폴더를 삭제하고 새로 만들지만, 실제 프로젝트에서는 기존 벡터 DB를 유지하거나 증분 업데이트하는 전략이 필요합니다.

### 5. 서버와 LLM 분리

Chroma 서버는 중앙에 두고, LLM은 사용자 로컬에서 실행하는 구조도 가능합니다. 이 방식은 문서 저장소 공유와 로컬 프라이버시 사이의 균형을 맞출 수 있습니다.


## 9.22 자주 발생하는 오류와 해결 방법

### 1. `chromadb` 설치 오류

해결:

```python
%pip install chromadb
```

설치 후 커널을 재시작합니다.

### 2. 기존 Chroma 저장소 삭제 실패

가능한 원인:

- 다른 프로세스가 `./chroma_store`를 사용 중
- 파일 권한 문제

해결:

- Chroma 서버 종료
- 노트북 커널 재시작
- 폴더를 수동 삭제

### 3. Chroma 서버 연결 실패

확인할 것:

- `chroma run --host 0.0.0.0 --port 8000 --path ./chroma_store` 실행 여부
- `SERVER_HOST`, `SERVER_PORT` 값
- 방화벽 또는 포트 차단

### 4. 컬렉션을 찾지 못함

가능한 원인:

- 서버에 해당 collection이 없음
- `collection_name`이 저장 시 이름과 다름
- 다른 `chroma_store` 경로를 보고 있음

### 5. 검색 품질이 낮음

가능한 원인:

- 임베딩 모델이 문서 언어에 맞지 않음
- 문서 청크 크기가 부적절함
- 저장 시 임베딩 모델과 검색 시 임베딩 모델이 다름
- collection에 원하는 문서가 저장되어 있지 않음


## 9.23 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- Chroma는 빠르게 시작하기 좋은 영속성 벡터 DB입니다.
- FAISS가 검색 라이브러리 성격이 강하다면, Chroma는 개발용 벡터 DB 사용성이 강합니다.
- `persist_directory`를 지정하면 로컬 폴더에 벡터 저장소를 유지할 수 있습니다.
- 긴 문서는 `RecursiveCharacterTextSplitter`로 청크 분할하는 것이 좋습니다.
- 문서 저장과 질문 검색에는 같은 임베딩 모델을 사용해야 합니다.
- Chroma 벡터스토어도 `as_retriever()`로 RAG 체인에 연결합니다.
- RAG 체인 구조는 FAISS와 Chroma가 거의 같습니다.
- Chroma는 서버로 실행하고 `HttpClient`로 원격 접속할 수도 있습니다.
- 서버형 Chroma는 중앙 벡터 DB와 로컬 LLM을 분리하는 구조에 적합합니다.

다음 장에서는 웹 URL 기반 RAG와 웹 문서 요약 실습을 정리합니다.
